<a href="https://colab.research.google.com/github/littlePanda2/ml-assignments/blob/main/assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 6 (Bonus): Linear Regression

**Course:** WBCS032-05 Introduction to Machine Learning  
**Student Names:** Sophie Schoerner
**Student Numbers:** s4136594

---

## Assignment Overview

In this assignment, you will implement linear regression as introduced and discussed in class, assuming an underlying dependence of the form $y = \mathbf{w} \cdot \mathbf{x}$ with $\mathbf{w}\in I\!\!R^{25}$.


You will work with the files `xtrain.csv`, `ytrain.csv`, `xtest.csv`, and `ytest.csv` with the data. These files contain two sets of 500 25-dimensional feature vectors and two sets of 500 continuous target labels.

## 1. Introduction (1 point)

Describe the goal of this assignment.

**Your answer here:**

The problem: We have inputs (x) and outputs (y), where the output variable y is generated by a linear combination of input features x and there’s a weight vector w connecting them. The task is to formulate a predictive model by calculating an estimate, w*, that minimizes the discrepancy between our predictions and the observed data. The ultimate goal is to find the minimum number of training samples (P) required for our estimated parameters to converge to the true underlying model.

The dataset: There are 500 training samples (xtrain.csv) and 500 test samples (xtest.csv) of the feature vectors (x). Each sample is a 25-dimensional vector, meaning there are 25 independent variables (features) influencing the outcome. The target labels (y) are continuous numerical values. Each y is generated by a "true" (but unknown) weight vector w such that y=w⋅x.

The algorithm: We’re using linear regression. We use a specific math shortcut called the Moore-Penrose Pseudoinverse. It finds the best possible weights even if the data is a bit messy/we don't have many samples.

The experiments: We’re testing two things. First, we’re plotting a bar chart of our calculated weights to see if they look like the "true" pattern. Second, we’re making a "Learning curve" graph to track how our error drops as we feed the model more and more data (from 30 samples up to all 500).

## 2. Methods (3 points)

### 2.1 Explain Linear Regression (0.5 points)

Explain the algorithm in a general manner.

**Your answer here:**

Linear Regression is a supervised learning algorithm used to model the relationship between some input features (x) and a continuous target variable (y). The model assumes that the output is a linear combination of the inputs, weighted by a parameter vector (w). Mathematically, it is expressed as y=w*x.

The goal of the algorithm is to find the optimal weight vector, w*, that minimizes the discrepancy between the model's predictions and the actual observed values in the dataset. To measure this discrepancy, we use a cost function called Mean Squared Error (MSE). The MSE calculates the average of the squared differences between the predicted and true values, squaring the "errors" ensures they are all positive and places a higher penalty on larger deviations.

To determine the optimal weights, we use the Normal Equation and the Moore-Penrose Pseudoinverse, and can solve for the weights in a single step. The pseudoinverse is very useful as it provides a stable solution even when the training data is limited or the input matrix is not full rank. This result, w*, represents the "best fit" parameters for the linear model based on the available training data.

### 2.2 Implementation (2.5 points)

You need to implement the linear regression **yourself**. Both the code quality and correctness will be graded.

*__Note:__* **Do not change the cell labels! Themis will use them to automatically grade your submission.**

In [61]:
# Load required libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

xtrain_path = 'xtrain.csv'
ytrain_path = 'ytrain.csv'
xtest_path  = 'xtest.csv'
ytest_path  = 'ytest.csv'

In [62]:
# Read the file containing the data

xtrain = np.array(pd.read_csv(xtrain_path, header=None))
ytrain = np.array(pd.read_csv(ytrain_path, header=None)).flatten()
xtest  = np.array(pd.read_csv(xtest_path,  header=None))
ytest  = np.array(pd.read_csv(ytest_path,  header=None)).flatten()

Implement linear regression in order to determine an estimate $\mathbf{w}^*$ from the first $P$ feature vectors and corresponding labels in the set `xtrain` and `ytrain`, respectively. To this end, construct the pseudoinverse solution discussed in class, which minimizes the Mean Squared Error (MSE):

$$ E_{train} =  \frac{1}{P}  
 \sum_{\mu=1}^P \, \frac{1}{2} \Big( \mathbf{w}^* \cdot \mathbf{x}_{train}^\mu - y_{train}^\mu \Big)^2.
$$

You can use available built-in functions or available code for the pseudoinverse (e.g. Python: `numpy.linalg.pinv`) but make sure that the matrix and vector dimensions are correct and that you obtain $\mathbf{w}^* \in I\!\!R^{25}.$  Unlike on the lecture slides we normalize the MSE here by $P$.
This does not  matter for the actual optimization, but  makes the resulting
MSE comparable for training
sets with different $P$ and the test error (see below).

With the resulting vector of parameters, also determine the corresponding test error
$$ E_{test} =  \frac{1}{1000}
 \sum_{\mu=1}^{500} \, \left( \mathbf{w}^* \cdot \mathbf{x}_{test}^\mu - y_{test}^\mu \right)^2
$$
which is always computed w.r.t. all of the $500$ samples in the test data set.

In [63]:
def linear_regression(data_x, data_y):
    '''
    Linear regression

    Args:
        data_x (ndarray): Data points.
        data_y (ndarray): output values

    Returns:
        w_star (ndarray): optimal parameters
    '''
    X = data_x
    y = data_y

    # pseudoinverse solution
    w_star = np.linalg.pinv(X) @ y

    return w_star

In [64]:
def MSE(data_x, data_y, w_star):
    '''
    Compute the MSE error

    Args:
        data_x (ndarray): Data points.
        data_y (ndarray): output values
        w_star (ndarray): the weights of linear regression

    Returns:
        error (float): the MSE error
    '''
    predictions = data_x @ w_star
    error = np.mean(0.5 * (predictions - data_y) ** 2)
    return error

## 3. Experimental Results (4 points)

### 3.1 Estimated weights

*__Note:__* This section **is graded** by Themis.

Implement the function below to generate bar graphs of the estimated parameter vector $\mathbf{w}^*$ for varying training set sizes $P$. The $x$-axis should correspond to the $25$ component indices and the $y$-axis should display the component $w^*_j$.

In [65]:
def plot_weights(xtrain, ytrain, p):
    """
    Plot the estimated parameter vector w* as a bar graph for a given training set size P.

    Args:
        xtrain (ndarray): Training feature vectors.
        ytrain (ndarray): Training labels.
        p (int): Number of training samples to use.
    """

    X = xtrain[:p]
    y = ytrain[:p]
    w_star = linear_regression(X, y)

    plt.figure()

    plt.bar(range(1, len(w_star) + 1), w_star)
    plt.xlabel("Component Index ($j$)", labelpad=3)
    plt.ylabel(r"$w_j^*$")
    plt.title(r"Estimated Parameter Vector $w^*$ for P = " + f"${p}$")
    plt.show()

### 3.2 Train and test error

*__Note:__* This section **is graded** by Themis.

Implement the function below to display both $E_{train}$ and $E_{test}$ for the following training set sizes: $P = 30, 40, 50, 75, 100, 200, 300, 400, 500$.

In [67]:
def plot_train_test_errors(xtrain, ytrain, xtest, ytest, p_values):
    """
    Plot training and test MSE errors vs training set size.

    Args:
        xtrain (ndarray): Training feature vectors.
        ytrain (ndarray): Training labels.
        xtest (ndarray): Test feature vectors.
        ytest (ndarray): Test labels.
        p_values (list): List of training set sizes to evaluate.
    """
    train_errors = []
    test_errors = []

    for p in p_values:
        X_train = xtrain[:p]
        y_train = ytrain[:p]

        w_star = linear_regression(X_train, y_train)

        train_errors.append(MSE(X_train, y_train, w_star))
        test_errors.append(MSE(xtest, ytest, w_star))

    plt.figure()
    plt.plot(p_values, train_errors, 'bo-', label="Training Error")
    plt.plot(p_values, test_errors, 'ro-', label="Test Error")

    plt.xlabel("Training Set Size (P)")
    plt.ylabel("Mean Squared Error")
    plt.legend()
    plt.title("Training and Test Errors vs. Training Set Size")
    plt.show()

Can you guess what the true vector $\mathbf{w}$ was that generated the data?

**Your answer here:**

## 4. Discussion (2 points)

Discuss your observations on the obtained results.

**Your answer here:**

The experiments show the transition from overfitting to model convergence as the training set size P increases.

Weight instability at small P: For P=30, the estimated weight vector w*
shows extreme fluctuations with very large magnitudes, meaning that the model is over-parameterized relative to the data. This causes the model to "memorize" the training samples, resulting in a training error of near zero but a big test error.

The learning curve: As P increases, the weight magnitudes stabilize significantly, settling into a pattern (see testcase p75). This stabilization is reflected in the Learning Curve, where the test error drops sharply and converges toward the training error.

Convergence and generalization: By P=500, the training and test errors are nearly identical, which suggests that the model has successfully identified the underlying linear relationship rather than local noise. The slight increase in training error as P grows might be a sign that the model is no longer perfectly interpolating the training points but is instead finding a generalized "best fit" for the entire distribution.

## Contribution

State your individual contribution.

**Your answer here:**

Sophie worked alone on this assignment.